# 04 · Modelling — Favorita

Modellvergleich auf dem gleichen chronologischen Split:

| Gruppe | Modell | Datenformat |
|---|---|---|
| Statistisch | SARIMAX | 1D-Serie pro Store |
| Statistisch | Prophet | 1D-Serie + exogene Regressoren |
| ML | XGBoost | 2D-Feature-Matrix |
| ML | LightGBM | 2D-Feature-Matrix |
| Deep Learning | PatchTST | Sequenzen via neuralforecast |
| Deep Learning | NHITS | Sequenzen + stat/hist exog via neuralforecast |

## 0 · Imports & Setup

In [3]:
import os
import sys
import itertools
import pathlib
import tempfile
import time
import json
from pathlib import Path

os.environ['CMDSTANPY_NOT_USE_POLARS'] = 'True'  

sys.path.append(os.path.abspath('../03_src'))
import joblib
import polars as pl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from neuralforecast import NeuralForecast
from neuralforecast.models import PatchTST
from neuralforecast.models import NHITS


from utilis import run_prophet, run_sarimax, evaluate, run_xgb, run_lgbm, to_nf_format
from config import (FINAL, TARGET_COL, FEATURE_COLS,EXOG_COLS, HIST_EXOG, STAT_EXOG,TRAIN_END, VAL_END, LOOKBACK, HORIZON)

RESULTS = FINAL / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

sns.set_style('whitegrid')
print('Setup abgeschlossen ✓')

Setup abgeschlossen ✓


## 1 · Daten laden & Train-Test Split

In [4]:
df = pl.read_parquet(FINAL / 'final_dataset.parquet')

train = df.filter(pl.col('date') <= TRAIN_END)
val   = df.filter((pl.col('date') > TRAIN_END) & (pl.col('date') <= VAL_END))
test  = df.filter(pl.col('date') > VAL_END)

print(f'Train: {train.shape}  {train["date"].min()} → {train["date"].max()}')
print(f'Val:   {val.shape}    {val["date"].min()}   → {val["date"].max()}')
print(f'Test:  {test.shape}   {test["date"].min()}  → {test["date"].max()}')

stores = train['store_nbr'].unique().sort().to_list()
print(f'Stores: {len(stores)}')

Train: (70114, 37)  2013-01-29 → 2016-12-31
Val:   (7965, 37)    2017-01-01   → 2017-05-31
Test:  (4104, 37)   2017-06-01  → 2017-08-15
Stores: 53


In [5]:
# Format 1: ML-Feature-Matrizen (für XGBoost & LightGBM)
X_train = train.select(FEATURE_COLS).to_numpy()
y_train = train.select(TARGET_COL).to_numpy().ravel()

X_val   = val.select(FEATURE_COLS).to_numpy()
y_val   = val.select(TARGET_COL).to_numpy().ravel()

X_test  = test.select(FEATURE_COLS).to_numpy()
y_test  = test.select(TARGET_COL).to_numpy().ravel()

print(f'X_train: {X_train.shape}  |  X_val: {X_val.shape}  |  X_test: {X_test.shape}')
print(f'Features ({len(FEATURE_COLS)}): {FEATURE_COLS}')

X_train: (70114, 33)  |  X_val: (7965, 33)  |  X_test: (4104, 33)
Features (33): ['store_type_enc', 'cluster', 'month', 'weekday', 'quarter', 'weekday_sin', 'weekday_cos', 'month_sin', 'month_cos', 'year', 'lag_1', 'lag_7', 'lag_14', 'lag_21', 'lag_28', 'rolling_mean_7_days', 'rolling_mean_14_days', 'rolling_mean_28_days', 'wow_growth', 'mom_growth', 'same_weekday_last_week', 'same_weekday_4weeks_ago', 'sales_vs_week_avg', 'wow_vs_month_avg', 'diff_1', 'diff_7', 'diff_28', 'oil_price', 'oil_price_ma7', 'is_national_holiday', 'is_day_before_holiday', 'is_day_after_holiday', 'is_holiday_window']


In [6]:
# Format 2: Format für Deep learnign Modelle
train_nf = to_nf_format(train)
val_nf   = to_nf_format(val)

static_df = (
    train.select(['store_nbr'] + STAT_EXOG)
         .unique(subset=['store_nbr'])
         .sort('store_nbr')
         .with_columns(pl.col('store_nbr').cast(pl.Utf8).alias('unique_id'))
         .select(['unique_id'] + STAT_EXOG)
         .with_columns([pl.col(c).cast(pl.Float32) for c in STAT_EXOG])
         .to_pandas()
)

print('NeuralForecast Format:')
print(train_nf.dtypes)
print(f'static_df shape: {static_df.shape}')
train_nf.head(3)

NeuralForecast Format:
unique_id                           str
ds                       datetime64[ms]
y                                 int64
store_type_enc                  float32
cluster                         float32
oil_price                       float32
is_national_holiday             float32
is_day_before_holiday           float32
is_day_after_holiday            float32
dtype: object
static_df shape: (53, 3)


,unique_id,ds,y,store_type_enc,cluster,oil_price,is_national_holiday,is_day_before_holiday,is_day_after_holiday
0,1,2013-01-30,1877,2.0,13.0,97.980003,0.0,0.0,0.0
1,1,2013-01-31,1707,2.0,13.0,97.650002,0.0,0.0,0.0
2,1,2013-02-01,1806,2.0,13.0,97.459999,0.0,0.0,0.0


In [7]:
# Trainingszeiten wegspeichern
training_times = {}

## 2 · Statistisch: SARIMAX


SARIMAX (Seasonal ARIMA with eXogenous regressors) wird als klassisches Zeitreihenmodell pro Filiale geschätzt. Die saisonale Ordnung (1,1,1,7)` bildet die dominante Wochensaisonalität ab (ACF Lag 7 ≈ 0.76).
Als exogene Regressoren werden Ölpreis und Feiertags-Flags verwendet. Das Modell dient als Baseline zur Einordnung der moderneren Ansätze. Trainingszeit wird erfasst und gespeichert.

In [8]:
t0 = time.time()
preds_sarimax = run_sarimax(
    train, val,
    stores=stores,
    exog_cols=EXOG_COLS
)
training_times['SARIMAX'] = round(time.time() - t0, 1)
preds_sarimax.write_parquet(RESULTS / 'val_sarimax.parquet')
print(f"SARIMAX ✓  {training_times['SARIMAX']}s")

SARIMAX:   0%|          | 0/53 [00:00<?, ?it/s]c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:836: ValueWarning: No sup

SARIMAX ✓  255.9s


## 3 · Statistisch: Prophet

Läuft pro Store separat. Exogene Regressoren: Ölpreis.  Das Modell läuft ebenfalls pro Store separat und dient als zweite Baseline neben SARIMAX

In [9]:
t0 = time.time()
preds_prophet = run_prophet(
    train, val,
    stores=stores,
    extra_regressors=['oil_price']
)
training_times['Prophet'] = round(time.time() - t0, 1)
preds_prophet.write_parquet(RESULTS / 'val_prophet.parquet')
print(f"Prophet ✓  {training_times['Prophet']}s")

Prophet Progress:   0%|          | 0/53 [00:00<?, ?it/s]21:16:38 - cmdstanpy - INFO - Chain [1] start processing
21:16:38 - cmdstanpy - INFO - Chain [1] done processing
Prophet Progress:   2%|▏         | 1/53 [00:00<00:42,  1.23it/s]21:16:38 - cmdstanpy - INFO - Chain [1] start processing
21:16:39 - cmdstanpy - INFO - Chain [1] done processing
Prophet Progress:   4%|▍         | 2/53 [00:01<00:28,  1.81it/s]21:16:39 - cmdstanpy - INFO - Chain [1] start processing
21:16:39 - cmdstanpy - INFO - Chain [1] done processing
Prophet Progress:   6%|▌         | 3/53 [00:01<00:23,  2.17it/s]21:16:39 - cmdstanpy - INFO - Chain [1] start processing
21:16:39 - cmdstanpy - INFO - Chain [1] done processing
Prophet Progress:   8%|▊         | 4/53 [00:01<00:20,  2.37it/s]21:16:40 - cmdstanpy - INFO - Chain [1] start processing
21:16:40 - cmdstanpy - INFO - Chain [1] done processing
Prophet Progress:   9%|▉         | 5/53 [00:02<00:19,  2.52it/s]21:16:40 - cmdstanpy - INFO - Chain [1] start processing
21

Prophet ✓  23.1s


## 4 · ML: XGBoost & LightGBM

Beide Modelle werden als globale Modelle auf der gesamten Feature-Matrix über alle 54 Filialen trainiert. Ein systematisches Grid Search über `max_depth` und `learning_rate` identifiziert die optimalen Hyperparameter anhand des Val-MAE. Das beste Modell wird gespeichert und die Parameter in `config.py` als `XGB_PARAMS` / `LGBM_PARAMS` hinterlegt, damit NB05 identische Konfigurationen für den Test-Set Run verwendet.

In [10]:
xgb_grid = {
    'max_depth': [5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1],
}


keys, values = zip(*xgb_grid.items())
permutations_dicts = [dict(zip(keys, v)) for v in itertools.product(*values)]

print(f"Starte XGBoost Tuning über {len(permutations_dicts)} Kombinationen mit run_xgb()...")

best_xgb_mae = float('inf')
best_xgb_params = {}
model_xgb = None
metrics_xgb = None

t0 = time.time()

for params in permutations_dicts:

    current_model, current_metrics = run_xgb(X_train, y_train, X_val, y_val, **params)
    
    try:
        current_mae = current_metrics['MAE']
    except (TypeError, KeyError):
        from sklearn.metrics import mean_absolute_error
        current_mae = mean_absolute_error(y_val, current_model.predict(X_val))
        
    if current_mae < best_xgb_mae:
        best_xgb_mae = current_mae
        best_xgb_params = params
        model_xgb = current_model
        metrics_xgb = current_metrics

training_times['XGBoost'] = round(time.time() - t0, 1)

print(f"Bestes Modell gefunden! Parameter: {best_xgb_params}")
print(f"Bester Validierungs-MAE: {round(best_xgb_mae, 2)}")

preds_xgb = (
    val.select(['store_nbr', 'date', TARGET_COL])
    .rename({TARGET_COL: 'y_true'})
    .with_columns(
        pl.Series('pred_xgb', model_xgb.predict(X_val).astype(float))
    )
)

# Ergebnisse sichern
preds_xgb.write_parquet(RESULTS / 'val_xgboost.parquet')
joblib.dump(model_xgb, RESULTS / 'model_xgb.pkl')

print(f"XGBoost ✓  {training_times['XGBoost']}s")
preds_xgb.head(5)

Starte XGBoost Tuning über 9 Kombinationen mit run_xgb()...
XGBoost Val           MAE=18.8  RMSE=37.2  MAPE=1.4%
XGBoost Val           MAE=13.3  RMSE=25.8  MAPE=0.9%
XGBoost Val           MAE=13.7  RMSE=26.4  MAPE=0.9%
XGBoost Val           MAE=11.8  RMSE=28.8  MAPE=0.9%
XGBoost Val           MAE=10.1  RMSE=24.9  MAPE=0.7%
XGBoost Val           MAE=10.7  RMSE=25.9  MAPE=0.7%
XGBoost Val           MAE=9.7  RMSE=25.7  MAPE=0.7%
XGBoost Val           MAE=8.6  RMSE=24.7  MAPE=0.6%
XGBoost Val           MAE=9.6  RMSE=26.1  MAPE=0.6%
Bestes Modell gefunden! Parameter: {'max_depth': 9, 'learning_rate': 0.05}
Bester Validierungs-MAE: 8.62
XGBoost ✓  57.1s


store_nbr,date,y_true,pred_xgb
i64,date,i64,f64
1,2017-01-02,516,539.421326
1,2017-01-03,1946,2109.21875
1,2017-01-04,1905,1903.567627
1,2017-01-05,1807,1797.639771
1,2017-01-06,1856,1854.046753


In [11]:
lgbm_grid = {
    'max_depth': [5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1],
}

keys, values = zip(*lgbm_grid.items())
permutations_dicts = [dict(zip(keys, v)) for v in itertools.product(*values)]

print(f"Starte LightGBM Tuning über {len(permutations_dicts)} Kombinationen mit run_lgbm()...")

best_lgbm_mae = float('inf')
best_lgbm_params = {}
model_lgbm = None
metrics_lgbm = None

t0 = time.time()

for params in permutations_dicts:
  
    current_model, current_metrics = run_lgbm(X_train, y_train, X_val, y_val, **params)
    

    try:
        current_mae = current_metrics['MAE']
    except (TypeError, KeyError):
        from sklearn.metrics import mean_absolute_error
        current_mae = mean_absolute_error(y_val, current_model.predict(X_val))
        
    if current_mae < best_lgbm_mae:
        best_lgbm_mae = current_mae
        best_lgbm_params = params
        model_lgbm = current_model
        metrics_lgbm = current_metrics

training_times['LightGBM'] = round(time.time() - t0, 1)

print(f"Bestes LightGBM Modell gefunden! Parameter: {best_lgbm_params}")
print(f"Bester Validierungs-MAE: {round(best_lgbm_mae, 2)}")

preds_lgbm = (
    val.select(['store_nbr', 'date', TARGET_COL])
    .rename({TARGET_COL: 'y_true'})
    .with_columns(
        pl.Series('pred_lgbm', model_lgbm.predict(X_val).astype(float))
    )
)

preds_lgbm.write_parquet(RESULTS / 'val_lightgbm.parquet')
joblib.dump(model_lgbm, RESULTS / 'model_lgbm.pkl')

print(f"LightGBM ✓  {training_times['LightGBM']}s")
preds_lgbm.head(5)

Starte LightGBM Tuning über 9 Kombinationen mit run_lgbm()...
LightGBM Val          MAE=18.6  RMSE=38.4  MAPE=1.4%
LightGBM Val          MAE=13.2  RMSE=27.8  MAPE=0.9%
LightGBM Val          MAE=13.8  RMSE=26.5  MAPE=0.9%
LightGBM Val          MAE=18.6  RMSE=38.1  MAPE=1.5%
LightGBM Val          MAE=12.5  RMSE=25.8  MAPE=0.8%
LightGBM Val          MAE=13.4  RMSE=25.5  MAPE=0.9%
LightGBM Val          MAE=18.6  RMSE=38.1  MAPE=1.5%
LightGBM Val          MAE=12.4  RMSE=25.9  MAPE=0.8%
LightGBM Val          MAE=13.1  RMSE=25.4  MAPE=0.9%
Bestes LightGBM Modell gefunden! Parameter: {'max_depth': 9, 'learning_rate': 0.05}
Bester Validierungs-MAE: 12.45
LightGBM ✓  35.9s


store_nbr,date,y_true,pred_lgbm
i64,date,i64,f64
1,2017-01-02,516,363.122757
1,2017-01-03,1946,1891.056324
1,2017-01-04,1905,1919.809232
1,2017-01-05,1807,1808.94165
1,2017-01-06,1856,1857.308743


## 5 · Deep Learning: PatchTST


PatchTST verarbeitet die Zeitreihe als globales Modell über alle 54 Filialen ohne filialspezifisches Training. Ein Grid Search über`patch_len`und `learning_rate` ∈ {1e-4, 5e-5} (8 Kombinationen) wird auf dem Val-Set evaluiert. Keine exogenen Features, da PatchTST in der verwendeten
Implementierung keine statischen oder historischen Regressoren unterstützt. standardisierung erfolgt pro Filiale via `scaler_type='standard'`.

HITS dekomponiert die Prognose hierarchisch in Stapel unterschiedlicher zeitlicher Auflösung. Im Gegensatz zu PatchTST werden hier statische Filialmerkmale (`store_type_enc`, `cluster`) sowie historische exogene Features (`oil_price`, Feiertags-Flags) als zusätzliche Eingaben verwendet. Die festen `pooling_widths=[2, 7, 14]` bilden
kurzfristige, wöchentliche und zweiwöchentliche Muster explizit ab. Grid Search über `learning_rate` ∈ {1e-3, 5e-4, 1e-4} und `batch_size` ∈ {64, 128} (6 Kombinationen).

In [12]:
import itertools

patch_tst_grid = {
    'patch_len': [7, 14],        
    'stride': [4, 7],             
    'learning_rate': [1e-4, 5e-5]
}

keys, values = zip(*patch_tst_grid.items())
permutations_dicts = [dict(zip(keys, v)) for v in itertools.product(*values)]

best_pt_mae = float('inf')
best_pt_params = {}
model_patchtst = None

t0 = time.time()

for params in permutations_dicts:
    print(f"Teste PatchTST Configuration -> PatchLen: {params['patch_len']}, Stride: {params['stride']}, LR: {params['learning_rate']}...")
    
    tuned_model = PatchTST(
        h=HORIZON,
        input_size=LOOKBACK,
        patch_len=params['patch_len'],
        stride=params['stride'],
        learning_rate=params['learning_rate'],
        encoder_layers=2,
        n_heads=8,
        hidden_size=64,
        linear_hidden_size=128,
        dropout=0.2,
        fc_dropout=0.2,
        scaler_type='standard',
        max_steps=150,           
        batch_size=64,
        early_stop_patience_steps=10,
        val_check_steps=25,
        random_seed=42,
    )
    
    nf_temp = NeuralForecast(models=[tuned_model], freq='D')
    
    nf_temp.fit(
        df=train_nf[['unique_id', 'ds', 'y']],
        val_size=HORIZON
    )
    
    preds_temp = nf_temp.predict().reset_index()
    preds_temp_pl = (
        pl.from_pandas(preds_temp)
        .rename({'unique_id': 'store_nbr', 'ds': 'date', 'PatchTST': 'pred'})
        .with_columns([pl.col('store_nbr').cast(pl.Int64), pl.col('date').cast(pl.Date)])
    )
    
    merged = preds_temp_pl.join(
        val.select(['store_nbr', 'date', TARGET_COL]).rename({TARGET_COL: 'y_true'}),
        on=['store_nbr', 'date'],
        how='inner'
    )
    
    current_mae = mean_absolute_error(merged['y_true'].to_numpy(), merged['pred'].to_numpy())
    print(f"--> Resultierender Validation-MAE: {round(current_mae, 2)}")
    
    if current_mae < best_pt_mae:
        best_pt_mae = current_mae
        best_pt_params = params
        model_patchtst = tuned_model

training_times['PatchTST'] = round(time.time() - t0, 1)

print(f"Tuning abgeschlossen! Beste PatchTST-Parameter: {best_pt_params}")
print(f"Minimaler Validierungs-MAE: {round(best_pt_mae, 2)}")


print("Generiere finalen Validierungs-Forecast mit den optimalen Parametern...")
model_patchtst.max_steps = 200

nf_patchtst = NeuralForecast(models=[model_patchtst], freq='D')
nf_patchtst.fit(
    df=train_nf[['unique_id', 'ds', 'y']],
    val_size=HORIZON
)
preds_patchtst = nf_patchtst.predict()

preds_patchtst_pl = (
    pl.from_pandas(preds_patchtst.reset_index())
    .rename({'unique_id': 'store_nbr', 'ds': 'date', 'PatchTST': 'pred_patchtst'})
    .with_columns([pl.col('store_nbr').cast(pl.Int64), pl.col('date').cast(pl.Date)])
    .join(val.select(['store_nbr', 'date', TARGET_COL]).rename({TARGET_COL: 'y_true'}),
          on=['store_nbr', 'date'], how='left')
)

preds_patchtst_pl.write_parquet(RESULTS / 'val_patchtst.parquet')
print(f"PatchTST ✓  Gesamtzeit (inkl. Tuning): {training_times['PatchTST']}s")
preds_patchtst_pl.head(5)

Seed set to 42


Teste PatchTST Configuration -> PatchLen: 7, Stride: 4, LR: 0.0001...


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.

  | Name         | Type              | Params | Mode  | FLOPs
-------------------------------------------------------------------
0 | loss         | MAE               | 0      | train | 0    
1 | padder_train | ConstantPad1d     | 0      | train | 0    
2 | scaler       | TemporalNorm      | 0      | train | 0    
3 | model        | PatchTST_backbone | 481 K  | train | 0    
-------------------------------------------------------------------
481 K     Trainable params
2         Non-trainable params
481 K     Total params
1.926     Total estimated model params size (MB)
65        Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=150` reached.
Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

PatchTST ✓  Gesamtzeit (inkl. Tuning): 3304.6s


index,store_nbr,date,pred_patchtst,y_true
i64,i64,date,f32,i64
0,1,2017-01-01,1171.582031,null
1,1,2017-01-02,400.955444,516
2,1,2017-01-03,1448.780151,1946
3,1,2017-01-04,1146.321533,1905
4,1,2017-01-05,1257.845581,1807


In [13]:
best_pt_params

{'patch_len': 14, 'stride': 4, 'learning_rate': 0.0001}

## 6 · Deep Learning: NHITS

Hierarchisches Interpolations-Modell.  
Unterstützt statische (`store_type_enc`, `cluster`) und historische exogene Features (`oil_price`, Feiertage).  

In [14]:
required_cols_nhits = ['unique_id', 'ds', 'y'] + HIST_EXOG

nhits_grid = {
    'learning_rate': [1e-3, 5e-4, 1e-4],
    'batch_size': [64, 128]
}

keys, values = zip(*nhits_grid.items())
permutations_dicts = [dict(zip(keys, v)) for v in itertools.product(*values)]


best_nh_mae = float('inf')
best_nh_params = {}
model_nhits = None

t0 = time.time()

for params in permutations_dicts:
    print(f"Teste NHITS -> LR: {params['learning_rate']}, Batch: {params['batch_size']}")
    
    tuned_model = NHITS(
        h=HORIZON,
        input_size=LOOKBACK,
        stat_exog_list=STAT_EXOG,
        hist_exog_list=HIST_EXOG,
        learning_rate=params['learning_rate'],
        batch_size=params['batch_size'],
        pooling_widths=[2, 7, 14],         
        scaler_type='standard',
        max_steps=150,                    
        early_stop_patience_steps=5,
        val_check_steps=10,
        random_seed=42
    )
    
    if 'pooling_widths' in tuned_model.trainer_kwargs:
        del tuned_model.trainer_kwargs['pooling_widths']
    
    nf_temp = NeuralForecast(models=[tuned_model], freq='D')
    
    nf_temp.fit(
        df=train_nf[required_cols_nhits],
        static_df=static_df,
        val_size=HORIZON
    )
    
    preds_temp = nf_temp.predict().reset_index()
    preds_temp_pl = (
        pl.from_pandas(preds_temp)
        .rename({'unique_id': 'store_nbr', 'ds': 'date', 'NHITS': 'pred'})
        .with_columns([pl.col('store_nbr').cast(pl.Int64), pl.col('date').cast(pl.Date)])
    )
    
    merged = preds_temp_pl.join(
        val.select(['store_nbr', 'date', TARGET_COL]).rename({TARGET_COL: 'y_true'}),
        on=['store_nbr', 'date'],
        how='inner'
    )
    
    current_mae = mean_absolute_error(merged['y_true'].to_numpy(), merged['pred'].to_numpy())
    print(f"--> Resultierender Validation-MAE: {round(current_mae, 2)}")
    
    if current_mae < best_nh_mae:
        best_nh_mae = current_mae
        best_nh_params = params
        model_nhits = tuned_model

print(f"\n🏆 Tuning abgeschlossen! Beste NHITS-Parameter: {best_nh_params}")
print(f"📉 Minimaler Validierungs-MAE: {round(best_nh_mae, 2)}")
print("Generiere finalen Validierungs-Forecast mit den optimalen Parametern...")

model_nhits_final = NHITS(
    h=HORIZON,
    input_size=LOOKBACK,
    stat_exog_list=STAT_EXOG,
    hist_exog_list=HIST_EXOG,
    learning_rate=best_nh_params['learning_rate'],
    batch_size=best_nh_params['batch_size'],
    pooling_widths=[2, 7, 14],     # Fest verbaut
    scaler_type='standard',
    max_steps=200,                  # Volle Länge für das finale Validierungs-Ergebnis
    early_stop_patience_steps=5,
    val_check_steps=10,
    random_seed=42
)

if 'pooling_widths' in model_nhits_final.trainer_kwargs:
    del model_nhits_final.trainer_kwargs['pooling_widths']

nf_nhits = NeuralForecast(models=[model_nhits_final], freq='D')
nf_nhits.fit(
    df=train_nf[required_cols_nhits],
    static_df=static_df,
    val_size=HORIZON
)

training_times['NHITS'] = round(time.time() - t0, 1)

preds_nhits = nf_nhits.predict()

preds_nhits_pl = (
    pl.from_pandas(preds_nhits.reset_index())
    .rename({'unique_id': 'store_nbr', 'ds': 'date', 'NHITS': 'pred_nhits'})
    .with_columns([pl.col('store_nbr').cast(pl.Int64), pl.col('date').cast(pl.Date)])
    .join(val.select(['store_nbr', 'date', TARGET_COL]).rename({TARGET_COL: 'y_true'}),
          on=['store_nbr', 'date'], how='left')
)

preds_nhits_pl.write_parquet(RESULTS / 'val_nhits.parquet')
print(f"NHITS ✓  Gesamtzeit (inkl. Tuning): {training_times['NHITS']}s")

with open(RESULTS / 'training_times.json', 'w') as f:
    json.dump(training_times, f, indent=2)

preds_nhits_pl.head(5)

Teste NHITS -> LR: 0.001, Batch: 64


Seed set to 42


c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\neuralforecast\tsdataset.py:129: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  x = torch.from_numpy(x)
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.

  | Name         | Type          | Params | Mode  | FLOPs
--------------------------------------------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

Validation: |          | 0/? [00:00<?, ?it/s]

Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

NHITS ✓  Gesamtzeit (inkl. Tuning): 691.5s


index,store_nbr,date,pred_nhits,y_true
i64,i64,date,f32,i64
0,1,2017-01-01,884.505737,null
1,1,2017-01-02,582.687073,516
2,1,2017-01-03,848.367615,1946
3,1,2017-01-04,416.67041,1905
4,1,2017-01-05,648.133484,1807


In [15]:
print('\n── Trainingszeiten (Sekunden) ──')
for m, t in training_times.items():
    print(f'  {m:<12} {t:>8.1f}s')

times_df = pl.DataFrame({
    "Modell": list(training_times.keys()),
    "Zeit_Sekunden": list(training_times.values())
})
times_df.write_parquet(RESULTS / 'trainingszeit.parquet')


── Trainingszeiten (Sekunden) ──
  SARIMAX         255.9s
  Prophet          23.1s
  XGBoost          57.1s
  LightGBM         35.9s
  PatchTST       3304.6s
  NHITS           691.5s


In [41]:
training_times

{'SARIMAX': 224.6,
 'Prophet': 26.7,
 'XGBoost': 65.4,
 'LightGBM': 26.4,
 'PatchTST': 6627.8,
 'NHITS': 6593.6}